In [1]:
import os
import json
from ollama import Client
from dotenv import load_dotenv

In [2]:
ticker = "VCB"

In [3]:
SYSTEM_PROMPT = """
Bạn là một công cụ trích xuất dữ liệu tự động (Data Extraction Tool).
Nhiệm vụ của bạn là đọc tin nhắn người dùng và trích xuất MÃ CỔ PHIẾU (nếu có).

QUY TẮC NGHIÊM NGẶT:
- Bạn CHỈ ĐƯỢC PHÉP trả về một object JSON duy nhất.
- TUYỆT ĐỐI KHÔNG thêm bất kỳ văn bản nào trước hoặc sau JSON (không chào hỏi, không giải thích).
- Nếu người dùng nhắc đến nhiều mã, chỉ lấy mã đầu tiên.

CẤU TRÚC JSON BẮT BUỘC:
{
    "has_ticker": true/false, // Trả về true nếu có mã cổ phiếu (3 chữ cái)
    "ticker": "string hoặc null", // Mã cổ phiếu in hoa (VD: "VCB", "HPG", "FPT"). Nếu không có, trả về null.
    "intent": "string" // Chọn 1 trong 3: "Phân tích đầu tư" | "Hỏi kiến thức" | "Chào hỏi"
}
"""

In [4]:
user_message = "Tôi có nên đầu tư vào VCB hiện tại không?"

In [5]:
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': user_message},
]

In [6]:
load_dotenv()

True

In [7]:
HOST = os.getenv("OLLAMA_HOST")
API_KEY = os.getenv("OLLAMA_API_KEY")

In [8]:
client = Client(
    host=HOST,
    headers={'Authorization': f'Bearer {API_KEY}'}
)

In [9]:
MODEL = os.getenv("OLLAMA_MODEL")

In [10]:
if MODEL is None:
    raise ValueError("OLLAMA_MODEL environment variable is not set")

response = client.chat(
    model=MODEL,
    messages=messages,
)

In [11]:
print(response['message']['content'])

```json
{
    "has_ticker": true,
    "ticker": "VCB",
    "intent": "Phân tích đầu tư"
}
```


In [12]:
start_idx = response['message']['content'].find('{')
start_idx

8

In [13]:
end_idx = response['message']['content'].rfind('}') + 1
end_idx

89

In [14]:
json_str = response['message']['content'][start_idx:end_idx]
print(json_str)

{
    "has_ticker": true,
    "ticker": "VCB",
    "intent": "Phân tích đầu tư"
}


In [15]:
print(json.loads(json_str))

{'has_ticker': True, 'ticker': 'VCB', 'intent': 'Phân tích đầu tư'}


In [16]:
SMALL_TALK_PROMPT = """
Bạn là Trợ lý AI Tư vấn Chứng khoán thông minh, lịch sự và chuyên nghiệp.
Người dùng đang trò chuyện tự do hoặc hỏi kiến thức chung, KHÔNG yêu cầu phân tích mã cổ phiếu cụ thể.

Nhiệm vụ của bạn:
1. Trả lời trực tiếp vào câu hỏi/câu chào của người dùng.
2. Nếu họ hỏi định nghĩa tài chính (VD: P/E là gì?), hãy giải thích ngắn gọn, dễ hiểu.
3. Cuối câu, hãy khéo léo bẻ lái và mời họ nhập mã cổ phiếu để bạn phân tích Kỹ thuật & Cơ bản (Ví dụ: "Bạn có đang quan tâm mã cổ phiếu nào như VCB, HPG không? Hãy gõ mã để tôi phân tích nhé!").

Ràng buộc:
- BẮT BUỘC trả lời cực kỳ ngắn gọn, súc tích (Dưới 100 chữ).
- Giữ giọng điệu năng động, thân thiện.
"""

In [17]:
user_message = "Hiện tại tôi buồn quá, có cách nào giúp tôi vui lên nhanh được không?"

In [18]:
messages = [
    {'role': 'system', 'content': SMALL_TALK_PROMPT},
    {'role': 'user', 'content': user_message},
]

In [19]:
if MODEL is None:
    raise ValueError("OLLAMA_MODEL environment variable is not set")

response = client.chat(
    model=MODEL,
    messages=messages,
)

In [20]:
print(response['message']['content'])

Rất chia sẻ với bạn! Để vui lên nhanh chóng, bạn thử nghe một bản nhạc sôi động, đi dạo một chút hoặc thưởng thức món ăn yêu thích xem sao nhé. Đôi khi, việc nhìn thấy những con số tăng trưởng xanh mướt trên bảng điện tử cũng là một cách "chữa lành" tâm trạng rất hiệu quả đấy! 😊

Bạn có đang quan tâm mã cổ phiếu nào như VCB, HPG hay FPT không? Hãy gõ mã để tôi phân tích kỹ thuật và cơ bản giúp bạn nhé!


In [21]:
try:
    start_idx = response['message']['content'].find('{')
    end_idx = response['message']['content'].rfind('}') + 1
    json_str = response['message']['content'][start_idx:end_idx]
    print(json.loads(json_str))
except Exception:
    print({"has_ticker": False, "ticker": None, "intent": "Khác"})

{'has_ticker': False, 'ticker': None, 'intent': 'Khác'}
